# 🏥 Pharmacy-Specific RAG System
### Retrieval-Augmented Generation for Pharmaceutical Decision Support

**Pipeline:** User Query → BioBERT Embedding → Pinecone Vector Search → Top-K Chunks → BART Summarisation → FDA Validation → Answer

---
Run each cell **top to bottom**. A Pinecone API key is optional — the demo works without it.

## Cell 1 — GPU Verification 🖥️

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU detected: {torch.cuda.get_device_name(0)}')
    print(f'   CUDA version : {torch.version.cuda}')
    print(f'   VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — running on CPU (slower but functional)')
    print('   Tip: Runtime → Change runtime type → GPU in Google Colab')

## Cell 2 — Clone Repository 📦

In [ ]:
import os

REPO_URL = 'https://github.com/Aqee232003/pharmacy-rag-system.git'
REPO_DIR = 'pharmacy-rag-system'

if os.path.exists(REPO_DIR):
    print(f'📁 Repository already cloned at {REPO_DIR}/')
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

print('\n✅ Repository ready — current directory:', os.getcwd())
!ls -la

## Cell 3 — Install Dependencies ⚙️

> This may take **3–5 minutes**. Output is hidden — a green checkmark appears when done.

In [ ]:
%%capture install_output
!pip install -q -r requirements.txt

# Verify critical packages
import importlib
packages = ['streamlit', 'transformers', 'torch', 'pinecone', 'langchain', 'requests']
for pkg in packages:
    try:
        mod = importlib.import_module(pkg.replace('-', '_'))
        ver = getattr(mod, '__version__', 'installed')
        print(f'  ✅ {pkg:30s} {ver}')
    except ImportError:
        print(f'  ❌ {pkg:30s} NOT FOUND')

print('\n✅ Dependencies installed successfully')

## Cell 4 — Set API Keys 🔑

| Key | Where to get it | Required? |
|-----|-----------------|----------|
| `PINECONE_API_KEY` | [pinecone.io](https://www.pinecone.io/) (free tier) | ❌ Optional |
| `NGROK_AUTH_TOKEN` | [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) | Required for Colab URL |

**The system works without Pinecone** (uses an in-memory vector store).

In [ ]:
import os
from getpass import getpass

# ── Pinecone (optional) ───────────────────────────────────────────────────
pinecone_key = getpass('Pinecone API key (press Enter to skip): ')
if pinecone_key.strip():
    os.environ['PINECONE_API_KEY'] = pinecone_key.strip()
    print('✅ Pinecone key set')
else:
    print('ℹ️  Pinecone skipped — will use in-memory vector store')

# ── ngrok (required for Colab public URL) ─────────────────────────────────
ngrok_key = getpass('ngrok auth token (press Enter to skip): ')
if ngrok_key.strip():
    os.environ['NGROK_AUTH_TOKEN'] = ngrok_key.strip()
    print('✅ ngrok token set')
else:
    print('ℹ️  ngrok skipped — app will only be accessible inside Colab')

print('\nConfiguration complete.')

## Cell 5 — Demo Without API Keys 🧪

Test the full pipeline using only the built-in sample pharmaceutical knowledge base. **No internet, no API keys needed.**

In [ ]:
import sys
sys.path.insert(0, '.')

from knowledge_base import PharmacyKnowledgeBase
from rag_pipeline import PharmacyRAGPipeline
from fda_validation import FDAValidator

print('🔬 Loading pipeline (BioBERT will download on first run ~400 MB)…')
pipeline = PharmacyRAGPipeline()

print('📦 Loading sample knowledge base…')
kb = PharmacyKnowledgeBase()
chunks = kb.get_sample_data()
n = pipeline.index_documents(chunks)
print(f'✅ Indexed {n} vectors')

# ── Run sample queries ────────────────────────────────────────────────────
demo_queries = [
    'What are the side effects of metformin?',
    'How do ACE inhibitors like lisinopril work?',
    'What are the risks of long-term ibuprofen use?',
]

validator = FDAValidator()

for q in demo_queries:
    print('\n' + '='*70)
    print(f'❓ Query: {q}')
    result = pipeline.query(q, top_k=3)
    print(f'\n💬 Answer:\n{result["answer"]}')
    print(f'\n📊 Retrieved {result["num_results"]} source(s)')
    for i, src in enumerate(result['sources'], 1):
        print(f'   [{i}] {src["source"]} (score={src["score"]:.3f})')

    # FDA validation
    fda = validator.get_validation_report(result['answer'], q)
    print(f'\n🏥 FDA: {fda["message"]}')

print('\n' + '='*70)
print('✅ Demo complete!')

## Cell 6 — Run Full Streamlit App 🚀

Launches the Streamlit web UI and creates a public ngrok tunnel so you can access it from your browser.

In [ ]:
import os
import threading
import time
from subprocess import Popen, PIPE

ngrok_token = os.environ.get('NGROK_AUTH_TOKEN', '')
port = 8501

# ── Start Streamlit in background ─────────────────────────────────────────
streamlit_process = Popen(
    ['streamlit', 'run', 'pharmacy_rag_app.py',
     '--server.port', str(port),
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=PIPE, stderr=PIPE
)

print(f'⏳ Starting Streamlit on port {port}…')
time.sleep(5)

if ngrok_token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(port)
    public_url = tunnel.public_url
    print(f'\n🌐 App is LIVE at: {public_url}')
    print('   Share this URL to access the app from any browser.')
else:
    print(f'\n⚠️  ngrok token not set.')
    print(f'   Access locally at: http://localhost:{port}')
    print('   Set NGROK_AUTH_TOKEN in Cell 4 to get a public URL.')

print('\n💡 The app will run until you interrupt this cell (■ Stop button).')

# Keep alive
try:
    streamlit_process.wait()
except KeyboardInterrupt:
    streamlit_process.terminate()
    print('\n🛑 Streamlit stopped.')

## Cell 7 — Programmatic Pipeline Test 🔬

Test individual pipeline components directly in the notebook.

In [ ]:
import sys
sys.path.insert(0, '.')

from rag_pipeline import PharmacyRAGPipeline
from document_processor import PharmacyDocumentProcessor
from fda_validation import FDAValidator
from knowledge_base import PharmacyKnowledgeBase
import json

# ── 1. Embedding test ─────────────────────────────────────────────────────
print('=== 1. BioBERT Embedding ===')
pipeline = PharmacyRAGPipeline()
emb = pipeline.create_embedding('metformin type 2 diabetes side effects')
print(f'Embedding dimension : {len(emb)}')
print(f'First 5 values      : {[round(x, 4) for x in emb[:5]]}')

# ── 2. Document processor test ────────────────────────────────────────────
print('\n=== 2. Document Processor ===')
processor = PharmacyDocumentProcessor()
sample_text = (
    'Metformin is a biguanide antidiabetic drug used as first-line therapy '
    'for type 2 diabetes mellitus. It reduces hepatic glucose production '
    'and improves insulin sensitivity.'
)
chunks = processor.process_text(sample_text, source='test')
print(f'Chunks produced: {len(chunks)}')
if chunks:
    print(f'First chunk: {chunks[0]["text"][:100]}…')

# ── 3. Knowledge base ─────────────────────────────────────────────────────
print('\n=== 3. Knowledge Base ===')
kb = PharmacyKnowledgeBase()
stats = kb.get_stats()
print(json.dumps(stats, indent=2))

# ── 4. Full RAG query ─────────────────────────────────────────────────────
print('\n=== 4. Full RAG Pipeline ===')
n = pipeline.index_documents(kb.get_sample_data())
print(f'Indexed {n} vectors')

result = pipeline.query('What is the mechanism of action of metformin?')
print(f'\nQuery  : {result["query"]}')
print(f'Answer : {result["answer"][:300]}…')
print(f'Sources: {result["num_results"]} chunks retrieved')

# ── 5. FDA validation ─────────────────────────────────────────────────────
print('\n=== 5. FDA Validation ===')
validator = FDAValidator()
report = validator.get_validation_report(result['answer'], result['query'])
print(f'Status    : {report["status"]}')
print(f'Confidence: {report["confidence_score"]}')
print(f'Message   : {report["message"]}')

print('\n✅ All pipeline components tested successfully!')